<h1>Project 2: Profitable app profiled for both App store and Google play</h1>


#### Step 1 Uploading csv (no pandas required)

In [1]:
#Importing Apple store dataset
import csv

with open("AppleStore.csv", encoding="utf-8") as f:
    reader = list(csv.reader(f))
    ios_header = reader[0]
    ios = reader[1:]

print("iOS rows:", len(ios))
print("iOS columns:", ios_header)
print(ios[:3])

iOS rows: 7197
iOS columns: ['', 'id', 'track_name', 'size_bytes', 'currency', 'price', 'rating_count_tot', 'rating_count_ver', 'user_rating', 'user_rating_ver', 'ver', 'cont_rating', 'prime_genre', 'sup_devices.num', 'ipadSc_urls.num', 'lang.num', 'vpp_lic']
[['1', '281656475', 'PAC-MAN Premium', '100788224', 'USD', '3.99', '21292', '26', '4', '4.5', '6.3.5', '4+', 'Games', '38', '5', '10', '1'], ['2', '281796108', 'Evernote - stay organized', '158578688', 'USD', '0', '161065', '26', '4', '3.5', '8.2.2', '4+', 'Productivity', '37', '5', '23', '1'], ['3', '281940292', 'WeatherBug - Local Weather, Radar, Maps, Alerts', '100524032', 'USD', '0', '188583', '2822', '3.5', '4.5', '5.0.0', '4+', 'Weather', '37', '5', '3', '1']]


In [2]:
#Importing the Google play (Android) dataset
with open("googleplaystore.csv", encoding="utf-8") as f:
    reader = list(csv.reader(f))
    android_header = reader[0]
    android = reader[1:]

print("Android rows:", len(android))
print("Android columns:", android_header)
print(android[:3])

Android rows: 10841
Android columns: ['App', 'Category', 'Rating', 'Reviews', 'Size', 'Installs', 'Type', 'Price', 'Content Rating', 'Genres', 'Last Updated', 'Current Ver', 'Android Ver']
[['Photo Editor & Candy Camera & Grid & ScrapBook', 'ART_AND_DESIGN', '4.1', '159', '19M', '10,000+', 'Free', '0', 'Everyone', 'Art & Design', 'January 7, 2018', '1.0.0', '4.0.3 and up'], ['Coloring book moana', 'ART_AND_DESIGN', '3.9', '967', '14M', '500,000+', 'Free', '0', 'Everyone', 'Art & Design;Pretend Play', 'January 15, 2018', '2.0.0', '4.0.3 and up'], ['U Launcher Lite – FREE Live Cool Themes, Hide Apps', 'ART_AND_DESIGN', '4.7', '87510', '8.7M', '5,000,000+', 'Free', '0', 'Everyone', 'Art & Design', 'August 1, 2018', '1.2.4', '4.0.3 and up']]


In [3]:
# There is a corrupted cell (see screenshot in Github) that shows a row with 12 columns but should be 13. 
print(len(android[10472])) 

12


In [4]:
#Deleting the row and validating this (it should be 13).
del android[10472]
print(len(android[10472])) 

13


Two datasets were used for this analysis: the Apple App Store dataset and the Google Play Store dataset. The datasets contain information on mobile applications, including category, user ratings, and install counts.

During initial inspection of the Google Play dataset, a malformed row with inconsistent column length was identified at index 10472. Since the row structure did not match the dataset schema, it was removed to maintain data integrity and prevent downstream analysis errors.

Key variables identified for analysis include app category/genre, number of installs (Android), and user ratings (iOS), which serve as proxies for user demand and engagement.

#### Step 2 Removing Duplicate entries & filtering to free, English apps

In [5]:
# Identifying duplicates for Google apps
duplicate_apps = []
unique_apps = []

for app in android:
    name = app[0]

    if name in unique_apps:
        duplicate_apps.append(name)
    else:
        unique_apps.append(name)
print("Number of duplicate apps:", len(duplicate_apps))
print("Examples:", duplicate_apps[:10])

Number of duplicate apps: 1181
Examples: ['Quick PDF Scanner + OCR FREE', 'Box', 'Google My Business', 'ZOOM Cloud Meetings', 'join.me - Simple Meetings', 'Box', 'Zenefits', 'Google Ads', 'Google My Business', 'Slack']


In [6]:
# Identifying duplicates for Apple apps
duplicate_apps = []
unique_apps = []

for app in ios:
    name = app[0]

    if name in unique_apps:
        duplicate_apps.append(name)
    else:
        unique_apps.append(name)
print("Number of iOS duplicate apps:", len(duplicate_apps))
print("Examples:", duplicate_apps[:10])

Number of iOS duplicate apps: 0
Examples: []


There are no duplicates for Apple so the cleaning will be focused on Google play.

In [7]:
#Building a review dictionary 
reviews_max = {}

for app in android:
    name = app[0]
    n_reviews = float(app[3])

    if name not in reviews_max:
        reviews_max[name] = n_reviews

    elif n_reviews > reviews_max[name]:
        reviews_max[name] = n_reviews


In [8]:
#Cleaning data
android_clean = []
already_added = []

for app in android:
    name = app[0]
    n_reviews = float(app[3])

    if (reviews_max[name] == n_reviews) and (name not in already_added):
        android_clean.append(app)
        already_added.append(name)

print("Clean android dataset:", len(android_clean))
    

Clean android dataset: 9659


In [9]:
#Filtering free apps
android_free = []

for app in android_clean:
    price = app[7]

    if price == '0':
        android_free.append(app)

print("Free Android apps:",len(android_free))

Free Android apps: 8905


In [10]:
ios_free = []

for app in ios:
    price = app[5]

    if price == '0':
        ios_free.append(app)

print("Free Apple apps:",len(ios_free))

Free Apple apps: 4056


In [11]:
# Filtering English Apps (some apps will have emojis, trademarks and accented letters, hence the non_ascii code)
def is_english(string):
    non_ascii = 0

    for character in string:
        if ord(character) > 127:
            non_ascii += 1

        if non_ascii > 3:
            return False
        else:
            return True


In [12]:
android_final = []

for app in android_free:
    name = app[0]
    
    if is_english(name):
        android_final.append(app)

print("Final android apps:", len(android_final))

Final android apps: 8905


In [13]:
ios_final = []

for app in ios_free:
    name = app[0]
    
    if is_english(name):
        ios_final.append(app)

print("Final iOS apps:", len(ios_final))

Final iOS apps: 4056


In [14]:
#Validating the result
non_english_ios = []

for app in ios_free:
    name = app[2]   # track_name

    if not is_english(name):
        non_english_ios.append(name)

print("Non-English free iOS apps:", len(non_english_ios))
print(non_english_ios[:10])

Non-English free iOS apps: 0
[]


In [15]:
non_english_android = []

for app in android_free:
    name = app[0]

    if not is_english(name):
        non_english_android.append(name)

print("Non-English free Android apps:", len(non_english_android))
print(non_english_android[:10])

Non-English free Android apps: 0
[]


The datasets were cleaned by removing duplicate applications, retaining only the entry with the highest review count to preserve the most complete record.

Since the company’s revenue model depends on advertising, only free applications were retained for analysis.

Finally, English-language applications were isolated using a character-based filtering function to focus the analysis on apps targeting a broad English-speaking audience. 

Duplicate checks were performed on both datasets. Significant duplicates were identified in the Google Play dataset and removed by retaining entries with the highest review count. The iOS dataset contained no meaningful duplicate records requiring removal. 

After filtering for free apps, the English-language filter did not reduce the iOS dataset further, indicating that all free iOS apps in the dataset met the English-language criteria used in this analysis.


#### Step 3 Identifying the most common genre

In [16]:
# Creating a frequency table
def freq_table(dataset, index):
    table = {}
    total = len(dataset)

    for row in dataset:
        key = row[index]

        if key in table:
            table[key] += 1
        else:
            table[key] = 1

    table_percentage = {}

    for key in table:
        percentage = (table[key] / total) * 100
        table_percentage[key] = percentage

    return table_percentage

In [21]:
# Creating a display sorted table function
def display_table(dataset, index):
    table = freq_table(dataset, index)
    table_display = []

    for key in table:
        key_value_as_tuple = (table[key], key)
        table_display.append(key_value_as_tuple)

    table_sorted = sorted(table_display, reverse=True)

    for entry in table_sorted:
        print(entry[1], ':', round(entry[0], 2), '%')

In [22]:
# Android category
display_table(android_final, 1)

FAMILY : 18.98 %
GAME : 9.7 %
TOOLS : 8.43 %
BUSINESS : 4.58 %
LIFESTYLE : 3.93 %
PRODUCTIVITY : 3.89 %
FINANCE : 3.68 %
MEDICAL : 3.51 %
SPORTS : 3.38 %
PERSONALIZATION : 3.31 %
COMMUNICATION : 3.23 %
HEALTH_AND_FITNESS : 3.07 %
PHOTOGRAPHY : 2.94 %
NEWS_AND_MAGAZINES : 2.83 %
SOCIAL : 2.65 %
TRAVEL_AND_LOCAL : 2.32 %
SHOPPING : 2.25 %
BOOKS_AND_REFERENCE : 2.18 %
DATING : 1.85 %
VIDEO_PLAYERS : 1.8 %
MAPS_AND_NAVIGATION : 1.41 %
FOOD_AND_DRINK : 1.24 %
EDUCATION : 1.17 %
ENTERTAINMENT : 0.95 %
LIBRARIES_AND_DEMO : 0.93 %
AUTO_AND_VEHICLES : 0.92 %
HOUSE_AND_HOME : 0.82 %
WEATHER : 0.8 %
EVENTS : 0.71 %
PARENTING : 0.65 %
ART_AND_DESIGN : 0.65 %
COMICS : 0.63 %
BEAUTY : 0.6 %


In [23]:
# Android genres
display_table(android_final, 9)

Tools : 8.42 %
Entertainment : 6.09 %
Education : 5.39 %
Business : 4.58 %
Lifestyle : 3.92 %
Productivity : 3.89 %
Finance : 3.68 %
Medical : 3.51 %
Sports : 3.45 %
Personalization : 3.31 %
Communication : 3.23 %
Action : 3.09 %
Health & Fitness : 3.07 %
Photography : 2.94 %
News & Magazines : 2.83 %
Social : 2.65 %
Travel & Local : 2.31 %
Shopping : 2.25 %
Books & Reference : 2.18 %
Simulation : 2.07 %
Dating : 1.85 %
Arcade : 1.84 %
Video Players & Editors : 1.77 %
Casual : 1.75 %
Maps & Navigation : 1.41 %
Food & Drink : 1.24 %
Puzzle : 1.12 %
Racing : 0.99 %
Role Playing : 0.93 %
Libraries & Demo : 0.93 %
Strategy : 0.92 %
Auto & Vehicles : 0.92 %
House & Home : 0.82 %
Weather : 0.8 %
Events : 0.71 %
Adventure : 0.69 %
Comics : 0.62 %
Art & Design : 0.61 %
Beauty : 0.6 %
Parenting : 0.49 %
Card : 0.45 %
Trivia : 0.43 %
Casino : 0.43 %
Educational;Education : 0.39 %
Board : 0.38 %
Educational : 0.37 %
Education;Education : 0.35 %
Word : 0.26 %
Casual;Pretend Play : 0.24 %
Music : 0

In [24]:
# ios genre
display_table(ios_final, 12)

Games : 55.65 %
Entertainment : 8.23 %
Photo & Video : 4.12 %
Social Networking : 3.53 %
Education : 3.25 %
Shopping : 2.98 %
Utilities : 2.69 %
Lifestyle : 2.32 %
Finance : 2.07 %
Sports : 1.95 %
Health & Fitness : 1.87 %
Music : 1.65 %
Book : 1.63 %
Productivity : 1.53 %
News : 1.43 %
Travel : 1.38 %
Food & Drink : 1.06 %
Weather : 0.76 %
Reference : 0.49 %
Navigation : 0.49 %
Business : 0.49 %
Catalogs : 0.22 %
Medical : 0.2 %


Frequency analysis revealed notable differences between the App Store and Google Play ecosystems.

The App Store is heavily dominated by entertainment-focused applications, particularly games (over 55%) and media-related categories. In contrast, Google Play contains a broader mix of utility and productivity-oriented applications, suggesting a more diverse platform ecosystem.

However, genre frequency alone does not indicate profitability or user engagement. Highly common categories may also represent saturated markets with intense competition. Therefore, further analysis is required to identify which genres attract the largest user bases.

#### Step 4 Analysing genres with the most users

In [29]:
#Android Weighted Category Analysis
categories_android = freq_table(android_final, 1)

android_category_analysis = []

for category in categories_android:
    total_installs = 0
    category_count = 0

    for app in android_final:
        category_app = app[1]

        if category_app == category:
            installs = app[5]

            installs = installs.replace(',', '')
            installs = installs.replace('+', '')

            total_installs += float(installs)
            category_count += 1

    avg_installs = total_installs/category_count
    android_category_analysis.append(
        (avg_installs, category_count, category)
    )


In [31]:
android_sorted = sorted(android_category_analysis, reverse = True)

for installs, count, category in android_sorted[:10]:
    print(
        category,
        '| Avg installs:', round(installs, 2),
        '| Apps:', count
)

COMMUNICATION | Avg installs: 38322625.7 | Apps: 288
VIDEO_PLAYERS | Avg installs: 24573948.25 | Apps: 160
SOCIAL | Avg installs: 23253652.13 | Apps: 236
PHOTOGRAPHY | Avg installs: 17772018.76 | Apps: 262
PRODUCTIVITY | Avg installs: 16738957.55 | Apps: 346
GAME | Avg installs: 15551995.89 | Apps: 864
TRAVEL_AND_LOCAL | Avg installs: 13984077.71 | Apps: 207
ENTERTAINMENT | Avg installs: 11640705.88 | Apps: 85
TOOLS | Avg installs: 10787009.95 | Apps: 751
NEWS_AND_MAGAZINES | Avg installs: 9401635.95 | Apps: 252


In [34]:
#ios Weighted Genre Analysis
genres_ios = freq_table(ios_final, 12)

ios_genre_analysis = []

for genre in genres_ios:

    total_ratings = 0
    genre_count = 0

    for app in ios_final:
        genre_app = app[12]

        if genre_app == genre:

            ratings = float(app[6])

            total_ratings += ratings
            genre_count += 1

    avg_ratings = total_ratings / genre_count

    ios_genre_analysis.append(
        (avg_ratings, genre_count, genre)
    )


In [36]:
ios_sorted = sorted(ios_genre_analysis, reverse=True)

for ratings, count, genre in ios_sorted[:10]:
    print(
        genre,
        '| Avg ratings:', round(ratings, 2),
        '| Apps:', count
    )

Reference | Avg ratings: 67447.9 | Apps: 20
Music | Avg ratings: 56482.03 | Apps: 67
Social Networking | Avg ratings: 53078.2 | Apps: 143
Weather | Avg ratings: 47220.94 | Apps: 31
Photo & Video | Avg ratings: 27249.89 | Apps: 167
Navigation | Avg ratings: 25972.05 | Apps: 20
Travel | Avg ratings: 20216.02 | Apps: 56
Food & Drink | Avg ratings: 20179.09 | Apps: 43
Sports | Avg ratings: 20128.97 | Apps: 79
Health & Fitness | Avg ratings: 19952.32 | Apps: 76


#### Step 5 Analysis and reccomendations

The analysis compared app categories across the Apple App Store and Google Play Store to identify genres with strong user demand and monetisation potential for free, ad-supported applications.

On Android, Communication, Video Players, and Social apps showed the highest average install counts. However, these categories are highly saturated and dominated by large established platforms, making market entry difficult for a smaller company.

On iOS, Reference apps demonstrated the highest average user engagement while maintaining a relatively small number of competing applications. Productivity-related categories on Android also showed strong install performance with broader practical appeal.

Based on these findings, the strongest opportunity would be to develop a productivity or educational reference application that provides practical value to users while supporting long-term engagement and advertising revenue. This category offers a more balanced combination of demand and competition compared to the highly saturated entertainment and social media market